In [ ]:
#  see here we used the alreadly calculated  temporalsentiment in the parallel file-- then only run this-- 
#  if you want to run both of the code together then do change the data handling logics below



#  this first block is for visualzing single user only
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# BASE PATH (FIXED FOR YOUR DATASET)
# ------------------------------------------------------------
BASE_DIR = "/kaggle/input/temporalsentiment"

# ------------------------------------------------------------
# 1. Load user-level metrics
# ------------------------------------------------------------
metrics = pd.read_csv(f"{BASE_DIR}/data/user_metrics.csv")

assert len(metrics) > 0, "user_metrics.csv is empty or missing."

print(f"[INFO] Loaded metrics for {len(metrics)} users")

# ------------------------------------------------------------
# 2. Define objective thresholds
# ------------------------------------------------------------
vol_thr = metrics["mean_volatility"].quantile(0.85)
drift_thr = metrics["drift_slope"].abs().quantile(0.85)

print(f"[INFO] Volatility threshold (85th %): {vol_thr:.4f}")
print(f"[INFO] Drift threshold (85th %): {drift_thr:.6f}")

# ------------------------------------------------------------
# 3. Select appropriate case-study users
# ------------------------------------------------------------
case_candidates = metrics[
    (metrics["mean_volatility"] >= vol_thr) |
    (metrics["drift_slope"].abs() >= drift_thr)
].copy()

assert len(case_candidates) > 0, "No suitable case-study users found."

print(f"[INFO] Suitable case-study users: {len(case_candidates)}")

# Prefer users with strongest volatility
case_candidates = case_candidates.sort_values(
    ["mean_volatility", "drift_slope"],
    ascending=False
)

user = case_candidates.iloc[0]["user"]

# Selection reason (for title/caption)
reason = (
    "high emotional volatility"
    if case_candidates.iloc[0]["mean_volatility"] >= vol_thr
    else "strong directional drift"
)

print(f"[INFO] Selected user: {user} ({reason})")

# ------------------------------------------------------------
# 4. Load archived time series for the selected user
# ------------------------------------------------------------
ts_path = f"{BASE_DIR}/timeseries/{user}_timeseries.csv"
df = pd.read_csv(ts_path, parse_dates=["datetime"])

assert len(df) > 0, "Time series CSV is empty."

# ------------------------------------------------------------
# 5. Journal-ready multi-panel case-study figure
# ------------------------------------------------------------
fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

# (a) Raw per-tweet sentiment
ax[0].plot(df["datetime"], df["sentiment"], alpha=0.4, linewidth=1)
ax[0].axhline(0, linestyle="--", color="gray")
ax[0].set_ylabel("Raw Sentiment")
ax[0].set_title("(a) Per-Tweet Sentiment (Short-Term Variability)")

# (b) Recency-weighted sentiment
ax[1].plot(df["datetime"], df["S_weighted"], linewidth=2)
ax[1].axhline(0, linestyle="--", color="gray")
ax[1].set_ylabel("Weighted Sentiment")
ax[1].set_title("(b) Recency-Weighted Temporal Sentiment")

# (c) Emotional volatility
ax[2].plot(df["datetime"], df["volatility"], linewidth=2, color="tab:red")
ax[2].set_ylabel("Volatility")
ax[2].set_xlabel("Time")
ax[2].set_title("(c) Emotional Volatility Over Time")

plt.suptitle(
    f"Case Study: User-Level Temporal Sentiment Dynamics ({reason})",
    y=0.98
)

plt.tight_layout()
plt.show()


In [ ]:

# this is for visualizing selected users
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "/kaggle/input/temporalsentiment"
metrics = pd.read_csv(f"{BASE_DIR}/data/user_metrics.csv")

# --------------------------------------------------
# Select representative users (unchanged logic)
# --------------------------------------------------
users_high_vol = metrics.sort_values(
    "mean_volatility", ascending=False
).head(2)["user"].tolist()

users_pos_drift = metrics.sort_values(
    "drift_slope", ascending=False
).head(1)["user"].tolist()

users_neg_drift = metrics.sort_values(
    "drift_slope", ascending=True
).head(1)["user"].tolist()

selected_users = list(set(users_high_vol + users_pos_drift + users_neg_drift))
print("Selected users:", selected_users)

# ==================================================
# (a) RECENCY-WEIGHTED SENTIMENT — SAVE FIRST
# ==================================================
plt.figure(figsize=(9,4))

for user in selected_users:
    df = pd.read_csv(
        f"{BASE_DIR}/timeseries/{user}_timeseries.csv",
        parse_dates=["datetime"]
    )
    plt.plot(df["datetime"], df["S_weighted"], linewidth=2, label=user)

plt.axhline(0, linestyle="--", color="gray")
plt.xlabel("Time")
plt.ylabel("Weighted Sentiment")
plt.title("(a) Recency-Weighted Sentiment — Multiple Users")
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()

plt.savefig("recency.pdf", format="pdf", bbox_inches="tight")
plt.show()
plt.close()

# ==================================================
# (b) EMOTIONAL VOLATILITY — SAVE FIRST
# ==================================================
plt.figure(figsize=(9,4))

for user in selected_users:
    df = pd.read_csv(
        f"{BASE_DIR}/timeseries/{user}_timeseries.csv",
        parse_dates=["datetime"]
    )
    plt.plot(df["datetime"], df["volatility"], linewidth=2, label=user)

plt.xlabel("Time")
plt.ylabel("Volatility")
plt.title("(b) Emotional Volatility — Multiple Users")
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()

plt.savefig("emotional.pdf", format="pdf", bbox_inches="tight")
plt.show()
plt.close()

# ==================================================
# (c) POPULATION DRIFT vs VOLATILITY — SAVE FIRST
# ==================================================
plt.figure(figsize=(6,4))

plt.scatter(
    metrics["mean_volatility"],
    metrics["drift_slope"],
    alpha=0.4,
    label="All users"
)

for user in selected_users:
    row = metrics[metrics["user"] == user].iloc[0]
    plt.scatter(
        row["mean_volatility"],
        row["drift_slope"],
        s=80,
        label=user
    )

plt.xlabel("Mean Volatility")
plt.ylabel("Drift Slope")
plt.title("(c) Drift vs Volatility (Population View)")
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()

plt.savefig("population.pdf", format="pdf", bbox_inches="tight")
plt.show()
plt.close()

print("✔ PDFs saved correctly: recency.pdf, emotional.pdf, population.pdf")
